# Structured Output Generation with OpenAI

This notebook provides a **detailed tutorial** on how to use OpenAI's structured output generation feature to extract structured data from text.

## What is Structured Output Generation?

Structured output generation allows you to define a **schema** (using Pydantic models) that specifies the exact format you want the LLM to return. Instead of getting free-form text that you need to parse, you get a validated, typed object that matches your schema.

**Key benefits:**
- **Type safety**: Responses are guaranteed to match your defined schema
- **No parsing needed**: The API returns structured data directly
- **Validation**: Pydantic validates the output automatically
- **Reliability**: Eliminates common issues with JSON parsing from raw text

## What we'll do in this notebook

We'll assess the quality of this very repository (https://github.com/thomas-sounack/TRIPOD-Code) using:
1. A **Pydantic model** defining the assessment schema
2. A **system prompt** instructing the model what to evaluate
3. **Repository extraction** to get the code content
4. **OpenAI's responses.parse API** to generate structured output

In [1]:
# =============================================================================
# STEP 1: Import Required Libraries
# =============================================================================
# 
# Pydantic is a data validation library that uses Python type hints.
# We use it to define the SCHEMA that the LLM must follow when generating output.
#
# - BaseModel: The base class for all Pydantic models
# - Field: Allows us to add metadata (descriptions) to each field
# - Optional: Marks fields that can be None
# - List: For fields that contain lists

from pydantic import BaseModel, Field
from typing import Optional, List

In [2]:
# =============================================================================
# STEP 2: Define the Output Schema (Pydantic Model)
# =============================================================================
#
# This is the MOST IMPORTANT part of structured output generation!
#
# We create a Pydantic model that defines EXACTLY what structure we want the LLM
# to produce. Each field has:
#   - A NAME (e.g., "is_empty")
#   - A TYPE (e.g., bool, str, Optional[bool])
#   - A DESCRIPTION (via Field(..., description="..."))
#
# The DESCRIPTION is crucial - it tells the LLM what each field should contain.
# Think of it as mini-instructions for each field.
#
# Key concepts:
#   - Use `...` (Ellipsis) to mark required fields
#   - Use `Optional[Type]` for fields that may be None
#   - The description should be clear and unambiguous
#
# See: https://docs.pydantic.dev/latest/concepts/models/
# See: https://platform.openai.com/docs/guides/structured-outputs


class RepoAssessment(BaseModel):
    """
    Schema for assessing the quality of a code repository.
    
    This model defines all the criteria we want the LLM to evaluate.
    Each field represents one aspect of repository quality.
    """
    
    # -------------------------------------------------------------------------
    # BASIC REPOSITORY STATUS
    # -------------------------------------------------------------------------
    
    is_empty: bool = Field(
        ...,  # Required field (cannot be None)
        description=(
            "Whether the repository is empty. Consider it empty if it contains no files, "
            "only empty files, or only a README file."
        ),
    )

    # -------------------------------------------------------------------------
    # README ASSESSMENT
    # -------------------------------------------------------------------------
    
    contains_readme: bool = Field(
        ...,
        description=(
            "Whether the repository contains usage/structure instructions "
            "(e.g., README.md/README.txt/README)."
        ),
    )
    
    # NOTE: This is a CONDITIONAL field - it should only be filled if contains_readme is True
    # We use Optional[bool] to allow None when the condition isn't met
    readme_purpose_and_outputs: Optional[bool] = Field(
        ...,
        description=(
            "If contains_readme is True, whether the README provides an overview of the "
            "repository purpose and expected outputs. Do not return anything if "
            "contains_readme is False."
        ),
    )

    # -------------------------------------------------------------------------
    # DEPENDENCIES & REQUIREMENTS
    # -------------------------------------------------------------------------
    
    contains_requirements: bool = Field(
        ...,
        description=(
            "Whether the repository specifies software dependencies either in a dedicated "
            "file (e.g., requirements.txt, environment.yml, pyproject.toml) or in the README."
        ),
    )
    
    requirements_dependency_versions: Optional[bool] = Field(
        ...,
        description=(
            "If contains_requirements is True, whether dependencies include version "
            "constraints (e.g., package==1.2.3, >=, ~=). Do not return anything if "
            "contains_requirements is False."
        ),
    )

    # -------------------------------------------------------------------------
    # LICENSE
    # -------------------------------------------------------------------------
    
    contains_license: bool = Field(
        ...,
        description="Whether the repository includes a license file describing usage permissions.",
    )

    # -------------------------------------------------------------------------
    # CODE QUALITY
    # -------------------------------------------------------------------------
    
    sufficient_code_documentation: bool = Field(
        ...,
        description=(
            "Whether the code contains sufficient inline comments/docstrings explaining "
            "key components so a user can understand the logic."
        ),
    )

    is_modular_and_structured: bool = Field(
        ...,
        description=(
            "Whether code is organized into modular, reusable components "
            "(functions/classes/modules) rather than a few long scripts."
        ),
    )

    implements_tests: bool = Field(
        ...,
        description=(
            "Whether the repository includes tests (unit/functional), test files/scripts, "
            "or meaningful assertions verifying expected behavior."
        ),
    )

    # -------------------------------------------------------------------------
    # REPRODUCIBILITY
    # -------------------------------------------------------------------------
    
    fixes_seed_if_stochastic: Optional[bool] = Field(
        ...,
        description=(
            "If the repository uses stochastic processes (e.g., random sampling, ML training), "
            "whether it sets fixed random seeds for reproducibility. Do not return anything "
            "if stochasticity is not applicable."
        ),
    )
    
    lists_hardware_requirements: bool = Field(
        ...,
        description="Whether hardware requirements (e.g., GPU/CPU/RAM) are stated anywhere.",
    )

    # -------------------------------------------------------------------------
    # CITATION & LINKING
    # -------------------------------------------------------------------------
    
    contains_link_to_paper: bool = Field(
        ...,
        description="Whether the repository includes a link (URL/DOI/arXiv/PubMed) to the paper.",
    )
    
    contains_citation: bool = Field(
        ...,
        description=(
            "Whether the repository provides a citation for the paper (e.g., BibTeX entry, "
            "CITATION.cff, or plain text citation)."
        ),
    )

    # -------------------------------------------------------------------------
    # DATA
    # -------------------------------------------------------------------------
    
    includes_data_or_sample: bool = Field(
        ...,
        description=(
            "Whether the repository includes the original dataset or a sample/demo dataset "
            "sufficient to run or demonstrate the code."
        ),
    )

    # -------------------------------------------------------------------------
    # FREE-TEXT FIELDS
    # -------------------------------------------------------------------------
    # 
    # Not everything fits into boolean fields! We can also have string fields
    # for open-ended responses.
    
    comments_and_explanations: Optional[str] = Field(
        ...,
        description=(
            "Additional comments about repository quality, strengths/weaknesses, and "
            "notable aspects not fully captured by the boolean fields."
        ),
    )

    # -------------------------------------------------------------------------
    # LIST FIELDS
    # -------------------------------------------------------------------------
    #
    # We can also have list fields for multiple values
    
    coding_languages: Optional[List[str]] = Field(
        ...,
        description=(
            "If the repository contains code, return all programming languages used as a list. "
            "For example, ['python', 'r', 'sql']. "
            "Do not return anything if there is no code in the repository."
        ),
    )


# Let's verify our model is valid by checking its fields
print("RepoAssessment schema fields:")
print("-" * 40)
for field_name, field_info in RepoAssessment.model_fields.items():
    field_type = field_info.annotation
    print(f"  {field_name}: {field_type}")

RepoAssessment schema fields:
----------------------------------------
  is_empty: <class 'bool'>
  contains_readme: <class 'bool'>
  readme_purpose_and_outputs: typing.Optional[bool]
  contains_requirements: <class 'bool'>
  requirements_dependency_versions: typing.Optional[bool]
  contains_license: <class 'bool'>
  sufficient_code_documentation: <class 'bool'>
  is_modular_and_structured: <class 'bool'>
  implements_tests: <class 'bool'>
  fixes_seed_if_stochastic: typing.Optional[bool]
  lists_hardware_requirements: <class 'bool'>
  contains_link_to_paper: <class 'bool'>
  contains_citation: <class 'bool'>
  includes_data_or_sample: <class 'bool'>
  comments_and_explanations: typing.Optional[str]
  coding_languages: typing.Optional[typing.List[str]]


In [3]:
# =============================================================================
# STEP 3: Define the System Prompt
# =============================================================================
#
# The system prompt provides HIGH-LEVEL instructions to the LLM about:
#   - What task it's performing
#   - How to interpret the input
#   - What criteria to use for evaluation
#
# KEY INSIGHT: The system prompt and the Pydantic field descriptions work TOGETHER:
#   - System prompt: Overall context and instructions
#   - Field descriptions: Specific guidance for each output field
#
# Tips for writing good system prompts:
#   1. Be specific about the task
#   2. Explain what input format to expect
#   3. Clarify any ambiguous criteria
#   4. Match the field names in your prompt to your Pydantic model


REPO_ASSESSMENT_PROMPT = """
You will be provided the tree of a repository and its code. Use it to assess the quality of the repository.

You should return a boolean for each of these categories:

- is_empty: is the repository empty? A repository is considered empty if it contains no files, only files that are empty, or only a README file.
- contains_readme: does the repository contain instructions on how the code is structured and how to use it (such as a README.md, README.txt or README file)?
- readme_purpose_and_outputs: (don't return anything for this field if contains_readme is false) do these instructions provide an overview of the purpose of the code repository, and its expected outputs?
- contains_requirements: does the repository specify the software dependencies used to run the code in a separate file (for example as a requirements.txt, environment.yml, or pyproject.toml file) or in the README file?
- requirements_dependency_versions: (don't return anything for this field if contains_requirements is false) does the requirements file specify dependency version requirements?
- contains_license: does the repository include a license file specifying how others can use this code?
- sufficient_code_documentation: does the code include sufficient inline documentation or comments explaining the purpose and functionality of key components of the code for a user to understand its logic?
- is_modular_and_structured: is the code organized into modular, reusable components using functions and classes where appropriate, rather than consisting of a single or a few long scripts?
- implements_tests: does the repository include unit tests or functional tests to verify that the code works as intended? This may include test scripts, test files, or embedded assertions that check whether inputs and outputs behave as expected.
- fixes_seed_if_stochastic: (If applicable, don't return anything if the repository doesn't use stochastic processes) if using stochastic processes (e.g., random number generation, machine learning models), is the repository setting fixed random seeds to ensure reproducibility?
- lists_hardware_requirements: are hardware requirements listed?
- contains_link_to_paper: does the repository contain a link to the paper it was used for?
- contains_citation: does the repository include a citation to the paper, in the format of a latex citation key or in plain text?
- includes_data_or_sample: does the repository include either the original dataset or a sample dataset for demonstration purposes?
- comments_and_explanations: provide additional comments and explanations regarding the repository's quality, strengths, weaknesses, or any notable aspects that may not be fully captured by the boolean assessments above.
- coding_languages: (if the repository contains code) list all programming languages used in the repository.
"""

print("System prompt loaded successfully!")
print(f"Prompt length: {len(REPO_ASSESSMENT_PROMPT)} characters")

System prompt loaded successfully!
Prompt length: 2844 characters


## Step 4: Prepare the Input Data

Before we can call the API, we need to prepare the **user prompt** - the actual content we want the LLM to analyze.

In our case, we want to assess a GitHub repository. We need to:
1. **Clone** the repository
2. **Extract** the file tree and code content
3. **Format** it as text that we can send to the API

This repository includes utilities in `src/repo_utils/` that handle this:
- `clone_and_extract_tree()`: Clones a repo and extracts its content
- Handles multiple repository sources (GitHub, GitLab, Zenodo, etc.)
- Applies token limits to avoid exceeding context windows

In [4]:
# =============================================================================
# STEP 4a: Import Repository Extraction Utilities
# =============================================================================
#
# We use our repo_utils module to clone and extract repository content.
# This handles the complexity of:
#   - Cloning from various sources (GitHub, GitLab, Zenodo, etc.)
#   - Building the file tree structure
#   - Reading and tokenizing file contents
#   - Applying token limits to stay within context windows

import sys
sys.path.insert(0, "../src")

from repo_utils import clone_and_extract_tree, RepoStatus

print("Repository utilities imported successfully!")

Repository utilities imported successfully!


In [5]:
# =============================================================================
# STEP 4b: Clone and Extract Repository Content
# =============================================================================
#
# We'll assess THIS REPOSITORY as our example!
# 
# The clone_and_extract_tree function:
#   - Clones the repository (or uses cached version if already cloned)
#   - Builds a tree structure of all files
#   - Reads and concatenates file contents
#   - Applies token limits to prevent context overflow
#
# Parameters:
#   - repo_url: The URL of the repository to analyze
#   - context_window: Maximum tokens to include (prevents exceeding LLM limits)
#   - clone_dir: Where to store the cloned repository

REPO_URL = "https://github.com/thomas-sounack/TRIPOD-Code"
CONTEXT_WINDOW = 390000  # Max tokens to include
CLONE_DIR = "../data/cloned_repos"

print(f"Extracting content from: {REPO_URL}")
print(f"Context window: {CONTEXT_WINDOW:,} tokens")
print("-" * 50)

# Clone and extract the repository
result = clone_and_extract_tree(
    repo_url=REPO_URL,
    context_window=CONTEXT_WINDOW,
    clone_dir=CLONE_DIR,
    verbose=True,
)

# Check if extraction was successful
if result.status == RepoStatus.OK:
    print(f"\n✅ Repository extracted successfully!")
    print(f"   Status: {result.status.value}")
    print(f"   Path: {result.repo_path}")
    print(f"   Output length: {len(result.output):,} characters")
    
    # Store the output for the next step
    user_prompt = result.output
else:
    print(f"\n❌ Extraction failed!")
    print(f"   Status: {result.status.value}")
    print(f"   Error: {result.error}")
    user_prompt = None

Extracting content from: https://github.com/thomas-sounack/TRIPOD-Code
Context window: 390,000 tokens
--------------------------------------------------
LICENSE
LICENSE 226
README.md
README.md 2244
.gitignore
.gitignore 1220
environment.yaml
environment.yaml 316
.env.example
.env.example 56
tests/test_llm_utils.py
tests/test_llm_utils.py 4463
tests/__init__.py
tests/__init__.py 9
tests/test_hello.py
tests/test_hello.py 35
tests/test_repo_utils.py
tests/test_repo_utils.py 4838
figures/figure7_language_distrib.png
figures/figure7_language_distrib.png 12
figures/figure5_repo_practices.png
figures/figure5_repo_practices.png 12
figures/appendix8_language_distribution_repositories.png
figures/appendix8_language_distribution_repositories.png 14
figures/figure3_code_sharing_journals.png
figures/figure3_code_sharing_journals.png 14
figures/figure2_code_sharing_over_time.png
figures/figure2_code_sharing_over_time.png 14
figures/figure4_code_sharing_countries.png
figures/figure4_code_sharing_coun

In [ ]:
user_prompt[-500:]  # Show the first 500 characters of the extracted content

'n\n[branch "main"]\n\tremote = origin\n\tmerge = refs/heads/main\n\n\n\nFILE: .git/shallow\n758525937a0dd2a508efd4e64eeca1fcd78c0009\n\n\n\nFILE: .git/HEAD\nref: refs/heads/main\n\n\n\nFILE: .git/description\nUnnamed repository; edit this file \'description\' to name the repository.\n\n\n\nFILE: .git/index\n\n\n\nFILE: .git/packed-refs\n# pack-refs with: peeled fully-peeled sorted \n758525937a0dd2a508efd4e64eeca1fcd78c0009 refs/remotes/origin/main\n\n\n\nFILE: notebooks/04_final_analysis.ipynb\n(REST OF THE REPOSITORY IS TRUNCATED)'

In [19]:
# =============================================================================
# STEP 4c: Preview the Extracted Content
# =============================================================================
#
# Let's take a look at what we extracted from the repository.
# This is what will be sent to the LLM as the "user prompt".

if user_prompt:
    # Show the first part of the extracted content
    preview_length = 3000000
    print("=" * 60)
    print("PREVIEW OF EXTRACTED REPOSITORY CONTENT")
    print("=" * 60)
    print(user_prompt[:preview_length])
    print("\n... [TRUNCATED FOR DISPLAY] ...")
    print(f"\nTotal content length: {len(user_prompt):,} characters")

PREVIEW OF EXTRACTED REPOSITORY CONTENT
Repository tree
{
  "url": "https://github.com/thomas-sounack/TRIPOD-Code",
  "tree": [
    {
      "name": "LICENSE",
      "children": null
    },
    {
      "name": "tests",
      "children": [
        {
          "name": "test_llm_utils.py",
          "children": null
        },
        {
          "name": "__init__.py",
          "children": null
        },
        {
          "name": "test_hello.py",
          "children": null
        },
        {
          "name": "test_repo_utils.py",
          "children": null
        }
      ]
    },
    {
      "name": "README.md",
      "children": null
    },
    {
      "name": "figures",
      "children": [
        {
          "name": "figure7_language_distrib.png",
          "children": null
        },
        {
          "name": "figure5_repo_practices.png",
          "children": null
        },
        {
          "name": "appendix8_language_distribution_repositories.png",
          "children":

## Step 5: Call the OpenAI API with Structured Output

Now comes the exciting part! We'll use OpenAI's `responses.parse()` method to:
1. Send our system prompt (instructions) and user prompt (repository content)
2. Request that the output conform to our `RepoAssessment` schema
3. Get back a **validated Pydantic object** directly!

### Key API Parameters:
- `model`: The model to use (e.g., "gpt-5.2-2025-12-11")
- `input`: List of messages (system + user prompts)
- `text_format`: The Pydantic class defining the output schema
- `truncation`: How to handle inputs that exceed context limits

In [15]:
# =============================================================================
# STEP 5a: Initialize the OpenAI Client
# =============================================================================
#
# The OpenAI client handles authentication and API communication.
# Make sure you have your API key set as an environment variable:
#   export OPENAI_API_KEY="your-api-key-here"
#
# Or create a .env file with:
#   OPENAI_API_KEY=your-api-key-here

from openai import OpenAI

# Initialize the client (automatically reads OPENAI_API_KEY from environment)
client = OpenAI()

print("✅ OpenAI client initialized successfully!")

✅ OpenAI client initialized successfully!


In [16]:
# =============================================================================
# STEP 5b: Call the API with Structured Output
# =============================================================================
#
# This is where the magic happens!
#
# The `responses.parse()` method:
#   1. Sends our prompts to the LLM
#   2. Instructs the model to output JSON matching our schema
#   3. Parses and validates the response automatically
#   4. Returns a Pydantic object we can use directly
#
# IMPORTANT: The `text_format` parameter is what enables structured output.
# This tells the API to constrain the model's output to match our schema.

MODEL_NAME = "gpt-4.1-2025-04-14"  # You can also use "gpt-4o", "gpt-4o-mini", etc.

print(f"🚀 Calling OpenAI API with model: {MODEL_NAME}")
print(f"   System prompt: {len(REPO_ASSESSMENT_PROMPT):,} chars")
print(f"   User prompt: {len(user_prompt):,} chars")
print("-" * 50)
print("⏳ Waiting for response (this may take a minute)...")

# Make the API call with structured output
response = client.responses.parse(
    model=MODEL_NAME,
    input=[
        # System message: Instructions for the model
        {"role": "system", "content": REPO_ASSESSMENT_PROMPT},
        # User message: The content to analyze
        {"role": "user", "content": user_prompt},
    ],
    # THIS IS THE KEY PARAMETER!
    # text_format tells the API to return structured output matching our Pydantic model
    text_format=RepoAssessment,
    # Auto-truncate if input is too long for the model's context window
    truncation="auto",
)

# The parsed output is available directly as a Pydantic object
assessment = response.output_parsed

print("\n✅ API call successful!")
print(f"   Response received and parsed into RepoAssessment object")

🚀 Calling OpenAI API with model: gpt-4.1-2025-04-14
   System prompt: 2,844 chars
   User prompt: 69,311 chars
--------------------------------------------------
⏳ Waiting for response (this may take a minute)...

✅ API call successful!
   Response received and parsed into RepoAssessment object


## Step 6: Explore the Results

Now we have our `assessment` object - a fully validated Pydantic model instance!

We can:
- Access fields directly with dot notation (`assessment.is_empty`)
- Convert to dictionary (`assessment.model_dump()`)
- Convert to JSON (`assessment.model_dump_json()`)
- Iterate over fields

In [17]:
# =============================================================================
# STEP 6a: Display the Assessment Results
# =============================================================================
#
# The `assessment` object is a Pydantic model instance.
# We can access its fields directly using dot notation.

print("=" * 70)
print("REPOSITORY ASSESSMENT RESULTS")
print(f"Repository: {REPO_URL}")
print("=" * 70)

# Helper function to format boolean values nicely
def format_bool(value):
    if value is None:
        return "N/A"
    return "✅ Yes" if value else "❌ No"

# Display results in a structured format
print("\n📊 BASIC STATUS")
print(f"   Is Empty: {format_bool(assessment.is_empty)}")

print("\n📖 README")
print(f"   Contains README: {format_bool(assessment.contains_readme)}")
print(f"   README has Purpose & Outputs: {format_bool(assessment.readme_purpose_and_outputs)}")

print("\n📦 DEPENDENCIES")
print(f"   Contains Requirements: {format_bool(assessment.contains_requirements)}")
print(f"   Has Version Constraints: {format_bool(assessment.requirements_dependency_versions)}")

print("\n📜 LICENSE")
print(f"   Contains License: {format_bool(assessment.contains_license)}")

print("\n💻 CODE QUALITY")
print(f"   Sufficient Documentation: {format_bool(assessment.sufficient_code_documentation)}")
print(f"   Modular & Structured: {format_bool(assessment.is_modular_and_structured)}")
print(f"   Implements Tests: {format_bool(assessment.implements_tests)}")

print("\n🔬 REPRODUCIBILITY")
print(f"   Fixes Random Seeds: {format_bool(assessment.fixes_seed_if_stochastic)}")
print(f"   Lists Hardware Requirements: {format_bool(assessment.lists_hardware_requirements)}")

print("\n📚 CITATION & LINKING")
print(f"   Contains Link to Paper: {format_bool(assessment.contains_link_to_paper)}")
print(f"   Contains Citation: {format_bool(assessment.contains_citation)}")

print("\n📁 DATA")
print(f"   Includes Data or Sample: {format_bool(assessment.includes_data_or_sample)}")

print("\n🗣️ PROGRAMMING LANGUAGES")
if assessment.coding_languages:
    print(f"   Languages: {', '.join(assessment.coding_languages)}")
else:
    print("   Languages: N/A")

REPOSITORY ASSESSMENT RESULTS
Repository: https://github.com/thomas-sounack/TRIPOD-Code

📊 BASIC STATUS
   Is Empty: ❌ No

📖 README
   Contains README: ✅ Yes
   README has Purpose & Outputs: ✅ Yes

📦 DEPENDENCIES
   Contains Requirements: ✅ Yes
   Has Version Constraints: ✅ Yes

📜 LICENSE
   Contains License: ✅ Yes

💻 CODE QUALITY
   Sufficient Documentation: ✅ Yes
   Modular & Structured: ✅ Yes
   Implements Tests: ✅ Yes

🔬 REPRODUCIBILITY
   Fixes Random Seeds: N/A
   Lists Hardware Requirements: ❌ No

📚 CITATION & LINKING
   Contains Link to Paper: ✅ Yes
   Contains Citation: ❌ No

📁 DATA
   Includes Data or Sample: ✅ Yes

🗣️ PROGRAMMING LANGUAGES
   Languages: python


In [18]:
# =============================================================================
# STEP 6b: View the LLM's Comments and Explanations
# =============================================================================
#
# The free-text field allows the LLM to provide additional context
# that doesn't fit into boolean fields.

print("=" * 70)
print("LLM COMMENTS AND EXPLANATIONS")
print("=" * 70)
print()
if assessment.comments_and_explanations:
    # Word-wrap the comments for better readability
    import textwrap
    wrapped = textwrap.fill(assessment.comments_and_explanations, width=70)
    print(wrapped)
else:
    print("No additional comments provided.")

LLM COMMENTS AND EXPLANATIONS

This repository demonstrates excellent open science practices: it has
a detailed README that describes the project purpose and the expected
workflow outputs, an MIT license, well-defined conda environment with
pinned versions, modular source code with clear organization, and
comprehensive unit and functional tests in the tests/ directory. Data
is provided for reproducibility in the data/ folder, along with
example notebook usage. There is a clear placeholder for a paper
citation, and a DOI is linked prominently. The only substantial
omissions are the lack of an explicit hardware requirements section,
and the citation information is not yet filled in. While stochasticity
could be present depending on how OpenAI models are called, there is
no clear evidence in the tree or test code that reproducibility via
fixed seeds is necessary in this specific repo workflow.


## Summary & Tips

### What We Learned

1. **Define your schema** using Pydantic models with descriptive fields
2. **Write a clear system prompt** that explains the task
3. **Prepare your input** (repository content in our case)
4. **Call `responses.parse()`** with `text_format` set to your Pydantic class
5. **Use the results** as a validated Python object

### Tips for Structured Output Generation

📝 **Schema Design:**
- Use clear, descriptive field names
- Write detailed descriptions in `Field(...)` - the LLM reads these!
- Use `Optional[Type]` for conditional fields
- Keep schemas focused - split large schemas into smaller ones if needed

🎯 **Prompting:**
- Match field names in your prompt to your schema
- Be explicit about edge cases
- Provide examples if the task is complex

⚡ **Performance:**
- Larger schemas = more tokens = higher cost
- Use `truncation="auto"` to handle long inputs gracefully
- Consider batching multiple assessments

🔧 **Debugging:**
- If results are unexpected, check your field descriptions first
- Print the raw response to see what the model returned
- Test with simpler schemas and build up complexity

### Resources

- [OpenAI Structured Outputs Guide](https://platform.openai.com/docs/guides/structured-outputs)
- [Pydantic Documentation](https://docs.pydantic.dev/)
- [This Repository's Source Code](https://github.com/thomas-sounack/TRIPOD-Code)